# Tag Redirection Mapper - Bali Touristic

Flow:
1. Upload CSV -> ambil kolom Request (A2 ke bawah)
2. Extract slug dari URL tag -> konversi ke keyword (- diganti spasi)
3. Search artikel via /wp/v2/posts?search={keyword} (ambil top 10)
4. Pilih artikel dengan judul paling mirip keyword (similarity score)
5. Export hasil ke CSV

In [ ]:
import requests
import csv
import time
from urllib.parse import urlparse
from requests.auth import HTTPBasicAuth
from difflib import SequenceMatcher

try:
    from google.colab import files, userdata
    IN_COLAB = True
    print('Berjalan di Google Colab')
except ImportError:
    IN_COLAB = False
    userdata = None
    print('Berjalan di luar Colab')

In [ ]:
WP_BASE_URL       = 'https://www.balitouristic.com'
WP_POSTS_ENDPOINT = WP_BASE_URL + '/wp-json/wp/v2/posts'

# Jumlah kandidat artikel yang diambil sebelum di-scoring
SEARCH_CANDIDATES = 10

# Minimum similarity score (0.0 - 1.0) agar dianggap relevan
# Turunkan nilainya jika terlalu banyak hasil kosong
MIN_SCORE = 0.3

if IN_COLAB and userdata:
    USERNAME = userdata.get('admin_bali')
    PASSWORD = userdata.get('pass_bali')
else:
    USERNAME = 'your_username'
    PASSWORD = 'your_password'

if not USERNAME or not PASSWORD:
    raise ValueError('Kredensial tidak ditemukan!')

AUTH = HTTPBasicAuth(USERNAME, PASSWORD)
print('Kredensial siap.')
print('Candidates per search:', SEARCH_CANDIDATES)
print('Min similarity score :', MIN_SCORE)

In [ ]:
if IN_COLAB:
    print('Silakan upload file CSV:')
    uploaded = files.upload()
    csv_input_filename = list(uploaded.keys())[0]
else:
    csv_input_filename = 'input.csv'

print('File dimuat:', csv_input_filename)

In [ ]:
requests_list = []

with open(csv_input_filename, mode='r', encoding='utf-8') as f:
    reader = csv.reader(f)
    rows = list(reader)

for row in rows[1:]:
    if row and row[0].strip():
        requests_list.append(row[0].strip())

print('Total Request ditemukan:', len(requests_list))
print('Preview 5 pertama:')
for r in requests_list[:5]:
    print(' -', r)

In [ ]:
def extract_slug_from_tag_url(tag_url):
    """Extract slug dari URL tag. Contoh: .../tag/wisata-bali/ -> wisata-bali"""
    path = urlparse(tag_url).path
    parts = [p for p in path.split('/') if p]
    return parts[-1] if parts else ''


def slug_to_keyword(slug):
    """Ubah slug ke keyword natural. Contoh: wisata-bali -> wisata bali"""
    return slug.replace('-', ' ')


def similarity(a, b):
    """Hitung similarity score antara dua string (0.0 - 1.0)."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def search_best_post(keyword, auth):
    """
    Search artikel via WP REST API, ambil top N kandidat,
    lalu pilih yang judulnya paling mirip dengan keyword.
    Return: (link, title, score) atau (None, None, 0)
    """
    try:
        response = requests.get(
            WP_POSTS_ENDPOINT,
            auth=auth,
            params={
                'search'  : keyword,
                'per_page': SEARCH_CANDIDATES,
                '_fields' : 'id,link,title'
            },
            timeout=10
        )

        if response.status_code != 200:
            print('  HTTP', response.status_code, 'untuk keyword:', keyword)
            return None, None, 0

        candidates = response.json()
        if not candidates:
            return None, None, 0

        # Scoring: bandingkan keyword vs judul tiap kandidat
        best_link  = None
        best_title = None
        best_score = 0.0

        for post in candidates:
            title = post.get('title', {}).get('rendered', '')
            link  = post.get('link', '')
            score = similarity(keyword, title)

            if score > best_score:
                best_score = score
                best_link  = link
                best_title = title

        if best_score >= MIN_SCORE:
            return best_link, best_title, round(best_score, 3)
        else:
            return None, best_title, round(best_score, 3)

    except requests.exceptions.RequestException as e:
        print('  Error:', e)
        return None, None, 0


print('Fungsi helper siap.')

In [ ]:
results = []
total = len(requests_list)
print('Memproses', total, 'request...')
print('')

for i, tag_url in enumerate(requests_list, start=1):
    prefix = '[' + str(i) + '/' + str(total) + ']'
    slug    = extract_slug_from_tag_url(tag_url)
    keyword = slug_to_keyword(slug)

    if not keyword:
        print(prefix, 'Slug tidak terdeteksi:', tag_url)
        results.append({
            'Request'    : tag_url,
            'Keyword'    : '',
            'Best Title' : '',
            'Score'      : '',
            'Destination': ''
        })
        continue

    destination, best_title, score = search_best_post(keyword, AUTH)

    if destination:
        print(prefix, '[OK]   keyword:', keyword)
        print('       score  :', score, '| title:', best_title)
        print('       dest   :', destination)
    else:
        print(prefix, '[MISS] keyword:', keyword, '| best score:', score, '| title:', best_title)

    print('')

    results.append({
        'Request'    : tag_url,
        'Keyword'    : keyword,
        'Best Title' : best_title or '',
        'Score'      : score,
        'Destination': destination or ''
    })

    time.sleep(0.3)

print('Selesai! Total diproses:', len(results))

In [ ]:
found     = sum(1 for r in results if r['Destination'])
not_found = len(results) - found

print('Ringkasan:')
print('  Berhasil     :', found)
print('  Tidak ketemu :', not_found)
print('  Total        :', len(results))

if not_found > 0:
    print('')
    print('Daftar yang tidak ketemu (pertimbangkan turunkan MIN_SCORE):')
    for r in results:
        if not r['Destination']:
            print(' - keyword:', r['Keyword'], '| best score:', r['Score'], '| title:', r['Best Title'])

In [ ]:
output_filename = 'tag_redirection_map.csv'

with open(output_filename, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Request', 'Keyword', 'Best Title', 'Score', 'Destination'])
    writer.writeheader()
    writer.writerows(results)

print('File disimpan:', output_filename)

if IN_COLAB:
    files.download(output_filename)
    print('Download otomatis dimulai...')